In [ ]:
from collections import Counter
from pathlib import Path

from astropy import table
import numpy as np
import pandas as pd

from ugdatalab import (
    GaiaData,
    WISEData,
    Compose,
    HighParallaxSnrCut, HighLatitudeCut, AddPhotometryColumns,
    LocalCut, StrictGCut, StrictBPRPCut,
    LindegrenC1Cut, LindegrenC2Cut,
    AttachInlierProbColumn,
    AddW2PhotometryColumns,
    WISEFiniteCut, MarreseOneToOneMatchCut, W2PhotometryQualityCut,
    fourier_fit,
    cross_validate,
)
from ugdatalab.methods.bayesian.likelihoods import LinearGaussianLikelihood
from ugdatalab.methods.bayesian.mcmc import nuts_sample

In [ ]:
# --- Lightcurve sample (cached) ---
lc_query = """
SELECT TOP 100 *
FROM gaiadr3.vari_rrlyrae
WHERE pf IS NOT NULL
  AND num_clean_epochs_g > 40
ORDER BY num_clean_epochs_g DESC
"""
rrlyrae = GaiaData(lc_query, include_lightcurve=True)

# --- Calibration pipeline (cached) ---
cal_query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
"""

_STRICT_PIPELINE = [
    HighParallaxSnrCut(), HighLatitudeCut(), AddPhotometryColumns(),
    LocalCut(), StrictGCut(), StrictBPRPCut(),
]
_C12_PIPELINE = _STRICT_PIPELINE + [LindegrenC1Cut(), LindegrenC2Cut()]

strict = GaiaData(cal_query, pipeline=Compose(_STRICT_PIPELINE))
c12 = GaiaData(cal_query, pipeline=Compose(_C12_PIPELINE + [AttachInlierProbColumn()]))
clean = GaiaData(cal_query, pipeline=Compose(_C12_PIPELINE + [AttachInlierProbColumn(prob_threshold=0.95)]))
rrlyrae_clean_data = clean.data

# --- Calibration sample from file ---
cal_data = np.load(Path("artifacts/rrlyrae_calibration_sample.npz"), allow_pickle=True)
calibration = table.Table({k: cal_data[k] for k in cal_data.files})
rrab_cal = calibration[calibration["best_classification"] == "RRab"]
rrc_cal = calibration[calibration["best_classification"] == "RRc"]

# --- Population and sample from files ---
pop_data = np.load(Path("artifacts/rrlyrae_reddening_population.npz"), allow_pickle=True)
population = table.Table({k: pop_data[k] for k in pop_data.files})

sample_data = np.load(Path("artifacts/rrlyrae_reddening_sample.npz"), allow_pickle=True)
rrlyrae_clean_map = table.Table({k: sample_data[k] for k in sample_data.files})

In [3]:
# =====================================================================
# SECTION 01: LIGHT CURVE ANALYSIS
# =====================================================================

lc = rrlyrae.lightcurves
_, first_idx = np.unique(lc["source_id"], return_index=True)
rows = lc[first_idx]

# --- Period recovery ---
pf = rows["pf"]
p_ls = rows["period_ls"]
p1_o = rows["p1_o"]

finite = np.isfinite(pf) & np.isfinite(p_ls)
frac_diff = np.abs(pf[finite] - p_ls[finite]) / pf[finite]
n_agree = int(np.sum(frac_diff < 0.01))

has_both = np.isfinite(pf) & np.isfinite(p1_o)
n_rrd = int(np.sum(has_both))

print(f"Period recovery: {n_agree} of {len(rows)} agree within 1%")
print(f"RRd double-mode pulsators: {n_rrd}")
if n_rrd > 0:
    ratio = p1_o[has_both] / pf[has_both]
    print(f"Median P1/P0 ratio: {np.median(ratio):.3f}")

# --- Fourier cross-validation (single target) ---
TARGET_ID = 4659759557323962752
lightcurve = lc[lc["source_id"] == TARGET_ID]
period_ls = lightcurve["period_ls"][0]

cv_result = cross_validate(
    lightcurve["g_transit_time"],
    lightcurve["g_transit_mag"],
    lightcurve["g_transit_mag_err"],
    lambda x, y, ye, k: fourier_fit(x, y, ye, period_ls, k),
    np.arange(1, 26),
)
best_K = cv_result.best_param

# --- Mean magnitude RMS ---
simple_g = rows["mean_g_transit_mag"]
fourier_g = rows["fourier_mean_g_mag"]
int_g = rows["int_average_g"]

valid_s = np.isfinite(simple_g) & np.isfinite(int_g)
valid_f = np.isfinite(fourier_g) & np.isfinite(int_g)

rms_epoch = float(np.sqrt(np.mean((simple_g[valid_s] - int_g[valid_s])**2)))
rms_fourier = float(np.sqrt(np.mean((fourier_g[valid_f] - int_g[valid_f])**2)))

print(f"\nOptimal Fourier order: K = {best_K}")
print(f"Epoch-mean RMS:   {rms_epoch:.4f} mag")
print(f"Fourier-mean RMS: {rms_fourier:.4f} mag")

Period recovery: 89 of 100 agree within 1%
RRd double-mode pulsators: 11
Median P1/P0 ratio: 0.744

Optimal Fourier order: K = 5
Epoch-mean RMS:   0.0282 mag
Fourier-mean RMS: 0.0139 mag


In [ ]:
# =====================================================================
# SECTION 02: CALIBRATION SAMPLE CONSTRUCTION
# =====================================================================

# --- C1/C2 quality cut cascade (each cut alone, applied to strict.data) ---
c1_only_data = LindegrenC1Cut()(strict.data)
c2_only_data = LindegrenC2Cut()(strict.data)

rows = [
    {"Stage": "Initial (Local + StrictG + StrictBPRP)", "N": len(strict.data)},
    {"Stage": "After C1 only", "N": len(c1_only_data)},
    {"Stage": "After C2 only", "N": len(c2_only_data)},
    {"Stage": "After C1 + C2", "N": len(c12.data)},
]
display(pd.DataFrame(rows))

n_removed = len(strict.data) - len(c12.data)
print(f"Removed by C1/C2: {n_removed} ({100*n_removed/len(strict.data):.1f}%)")

# --- Spatial rejection rates ---
l_all = np.array(strict.data["l"], dtype=float)
b_all = np.array(strict.data["b"], dtype=float)
ids_kept = set(np.array(c12.data["source_id"]))
removed_mask = np.array([sid not in ids_kept for sid in strict.data["source_id"]])

for name, l0, b0, radius in [("LMC (10° radius)", 280, -33, 10), ("SMC (6° radius)", 303, -44, 6)]:
    dist = np.sqrt((l_all - l0)**2 + (b_all - b0)**2)
    in_region = dist < radius
    n_in = int(np.sum(in_region))
    n_removed_region = int(np.sum(in_region & removed_mask))
    if n_in > 0:
        print(f"{name}: {n_removed_region}/{n_in} = {100*n_removed_region/n_in:.1f}% rejected")

lmc_or_smc = (np.sqrt((l_all-280)**2+(b_all+33)**2) < 10) | (np.sqrt((l_all-303)**2+(b_all+44)**2) < 6)
removed_elsewhere = int(np.sum(removed_mask & ~lmc_or_smc))
total_elsewhere = int(np.sum(~lmc_or_smc))
print(f"Elsewhere: {removed_elsewhere}/{total_elsewhere} = {100*removed_elsewhere/total_elsewhere:.1f}% rejected")

# --- PL residual RMS at each stage ---
stages = [
    ("Initial (strict)", strict.data),
    ("After C1+C2", c12.data),
    ("After Deoutlier", rrlyrae_clean_data),
]
rms_values = {}
for label, data_table in stages:
    resid_list = []
    for cls in ["RRab", "RRc"]:
        sub = data_table[data_table["best_classification"] == cls]
        if len(sub) == 0:
            continue
        log_p = np.log10(sub["rrlyrae_representative_period"])
        M = np.array(sub["M_G"], dtype=float)
        coeffs = np.polyfit(log_p, M, 1)
        resid = M - np.polyval(coeffs, log_p)
        resid_list.append(resid)
    all_resid = np.concatenate(resid_list)
    rms = float(np.sqrt(np.mean(all_resid**2)))
    n_outlier = int(np.sum(np.abs(all_resid) > 1.0))
    rms_values[label] = rms
    print(f"\n{label} (N={len(data_table)}): RMS = {rms:.3f} mag, |resid| > 1 mag: {n_outlier}")

rms_init = rms_values["Initial (strict)"]
rms_c12 = rms_values["After C1+C2"]
rms_clean = rms_values["After Deoutlier"]
print(f"\nC1/C2 RMS improvement: {100*(1 - rms_c12/rms_init):.0f}%")
print(f"Mixture model RMS improvement (from C1+C2): {100*(1 - rms_clean/rms_c12):.0f}%")

# --- Final composition ---
counts = Counter(rrlyrae_clean_data["best_classification"])
print(f"\nFinal sample: {len(rrlyrae_clean_data)} stars")
for cls in ["RRab", "RRc", "RRd"]:
    print(f"  {cls}: {counts.get(cls, 0)}")

In [5]:
# =====================================================================
# SECTION 03b: G-BAND PERIOD-LUMINOSITY PARAMETERS
# =====================================================================

def build_pl_arrays(data, mag_col, err_col):
    period = np.array(data["rrlyrae_representative_period"])
    period_err = np.array(data["rrlyrae_representative_period_error"])
    log_p = np.log10(period)
    sigma_logp = period_err / (period * np.log(10))
    return log_p - np.mean(log_p), np.array(data[mag_col]), np.array(data[err_col]), sigma_logp

def _posterior_table(result, label):
    s = result.samples
    med = np.median(s, axis=0)
    lo = np.percentile(s, 15.87, axis=0)
    hi = np.percentile(s, 84.13, axis=0)
    rows = []
    names = ["a (slope)", "b (intercept)", "log10(σ)"]
    for i, name in enumerate(names):
        rows.append({"Parameter": name, "Median": f"{med[i]:.4f}",
                      "-1σ": f"{med[i]-lo[i]:.4f}", "+1σ": f"{hi[i]-med[i]:.4f}"})
    sig_samples = 10.0 ** s[:, 2]
    med_sig = np.median(sig_samples)
    lo_sig = np.percentile(sig_samples, 15.87)
    hi_sig = np.percentile(sig_samples, 84.13)
    rows.append({"Parameter": "σ_scatter", "Median": f"{med_sig:.4f}",
                  "-1σ": f"{med_sig-lo_sig:.4f}", "+1σ": f"{hi_sig-med_sig:.4f}"})
    return pd.DataFrame(rows)

# --- RRab G-band fit ---
rrab_optical_result = nuts_sample(LinearGaussianLikelihood(*build_pl_arrays(rrab_cal, "M_G", "sigma_M")))
rrc_optical_result = nuts_sample(LinearGaussianLikelihood(*build_pl_arrays(rrc_cal, "M_G", "sigma_M")))

print(f"=== RRab G-band PL (N = {len(rrab_cal)}) ===")
print(f"NUTS: 1,000 tune, 2,000 draws")
display(_posterior_table(rrab_optical_result, "RRab"))

print(f"\n=== RRc G-band PL (N = {len(rrc_cal)}) ===")
display(_posterior_table(rrc_optical_result, "RRc"))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, log10_sig]
Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 22 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, log10_sig]
Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 12 seconds.


=== RRab G-band PL (N = 559) ===
NUTS: 1,000 tune, 2,000 draws


,Parameter,Median,-1σ,+1σ
0,a (slope),-2.2307,0.1255,0.1238
1,b (intercept),0.6524,0.0087,0.0087
2,log10(σ),-0.7562,0.0183,0.0183
3,σ_scatter,0.1753,0.0072,0.0075



=== RRc G-band PL (N = 316) ===


,Parameter,Median,-1σ,+1σ
0,a (slope),-2.1925,0.1824,0.1738
1,b (intercept),0.5735,0.0110,0.0112
2,log10(σ),-0.7757,0.0249,0.0240
3,σ_scatter,0.1676,0.0093,0.0095


In [ ]:
# =====================================================================
# SECTION 03c: WISE W2 CROSS-MATCH & INFRARED PL
# =====================================================================

wise_query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
LEFT OUTER JOIN gaiadr3.allwise_best_neighbour AS bn
    ON vr.source_id = bn.source_id
LEFT OUTER JOIN gaiadr3.allwise_neighbourhood AS nh
    ON bn.source_id = nh.source_id
   AND bn.allwise_oid = nh.allwise_oid
LEFT OUTER JOIN gaiadr1.allwise_original_valid AS aw
    ON bn.allwise_oid = aw.allwise_oid
"""

wise_unfiltered = WISEData(wise_query)
wise_filtered = WISEData(wise_query, pipeline=Compose([
    HighParallaxSnrCut(),
    HighLatitudeCut(),
    AddW2PhotometryColumns(),
    WISEFiniteCut(),
    MarreseOneToOneMatchCut(),
    W2PhotometryQualityCut(),
]))

optical_ids = calibration["source_id"]
wise_ids = wise_filtered.data["source_id"]
subset_ids = np.isin(wise_ids, optical_ids)
wise_data = wise_filtered.data[subset_ids]

rrab_infrared = wise_data[wise_data["best_classification"] == "RRab"]
rrc_infrared = wise_data[wise_data["best_classification"] == "RRc"]

# --- Cross-match summary ---
print(f"Full Gaia×AllWISE join: {len(wise_unfiltered.data)} rows")
print(f"After WISE quality cuts: {len(wise_filtered.data)} rows")
print(f"Matched to calibration sample: {len(wise_data)} stars")

counts = Counter(wise_data["best_classification"])
for cls in ["RRab", "RRc", "RRd"]:
    print(f"  {cls}: {counts.get(cls, 0)}")

n_for_pl = counts.get("RRab", 0) + counts.get("RRc", 0)
print(f"Stars used for PL analysis (RRab + RRc): {n_for_pl}")

# --- IR PL fits ---
rrab_infrared_result = nuts_sample(LinearGaussianLikelihood(*build_pl_arrays(rrab_infrared, "M_W2", "sigma_M_W2")))
rrc_infrared_result = nuts_sample(LinearGaussianLikelihood(*build_pl_arrays(rrc_infrared, "M_W2", "sigma_M_W2")))

def _posterior_row(result, label):
    s = result.samples
    med = np.median(s, axis=0)
    lo = np.percentile(s, 15.87, axis=0)
    hi = np.percentile(s, 84.13, axis=0)
    sig_samples = 10.0 ** s[:, 2]
    return {
        "Band": label,
        "a (slope)": f"{med[0]:.4f} ± {(hi[0]-lo[0])/2:.4f}",
        "b (intercept)": f"{med[1]:.4f} ± {(hi[1]-lo[1])/2:.4f}",
        "σ_scatter": f"{np.median(sig_samples):.4f} ± {(np.percentile(sig_samples,84.13)-np.percentile(sig_samples,15.87))/2:.4f}",
    }

print()
rows = [
    _posterior_row(rrab_optical_result, "G RRab"),
    _posterior_row(rrc_optical_result, "G RRc"),
    _posterior_row(rrab_infrared_result, "W2 RRab"),
    _posterior_row(rrc_infrared_result, "W2 RRc"),
]
display(pd.DataFrame(rows))

# --- Slope differences ---
for cls, opt, ir in [("RRab", rrab_optical_result, rrab_infrared_result),
                      ("RRc", rrc_optical_result, rrc_infrared_result)]:
    da = np.median(ir.samples[:, 0]) - np.median(opt.samples[:, 0])
    da_err = np.sqrt(np.std(ir.samples[:, 0])**2 + np.std(opt.samples[:, 0])**2)
    print(f"Δa_{cls} (W2 - G) = {da:.2f} ± {da_err:.2f}")

In [7]:
# =====================================================================
# SECTION 03d: PERIOD-COLOR RELATIONS & g_absorption COMPARISON
# =====================================================================

def build_pc_arrays(data):
    period = np.array(data["rrlyrae_representative_period"])
    period_err = np.array(data["rrlyrae_representative_period_error"])
    log_p = np.log10(period)
    bp_rp = np.array(data["bp_rp"])
    snr_bp = np.array(data["phot_bp_mean_flux_over_error"])
    snr_rp = np.array(data["phot_rp_mean_flux_over_error"])
    sigma_color = (2.5 / np.log(10)) * np.sqrt(1 / snr_bp**2 + 1 / snr_rp**2)
    sigma_logp = period_err / (period * np.log(10))
    return log_p - np.mean(log_p), bp_rp, sigma_color, sigma_logp

rrab_pc_result = nuts_sample(LinearGaussianLikelihood(*build_pc_arrays(rrab_cal)))
rrc_pc_result = nuts_sample(LinearGaussianLikelihood(*build_pc_arrays(rrc_cal)))

def _pc_summary(result, label, cal_data):
    s = result.samples
    med = np.median(s, axis=0)
    lo = np.percentile(s, 15.87, axis=0)
    hi = np.percentile(s, 84.13, axis=0)
    sig_samples = 10.0 ** s[:, 2]
    return {
        "Class": f"{label} (N={len(cal_data)})",
        "a (slope)": f"{med[0]:+.4f} \u00b1 {(hi[0]-lo[0])/2:.4f}",
        "b (intercept)": f"{med[1]:+.4f} \u00b1 {(hi[1]-lo[1])/2:.4f}",
        "\u03c3_color": f"{np.median(sig_samples):.4f} \u00b1 {(np.percentile(sig_samples,84.13)-np.percentile(sig_samples,15.87))/2:.4f}",
    }

rows = [_pc_summary(rrab_pc_result, "RRab", rrab_cal),
        _pc_summary(rrc_pc_result, "RRc", rrc_cal)]
display(pd.DataFrame(rows))

# --- Gaia g_absorption comparison (RRab only) ---
rrab_pop = population[population["best_classification"] == "RRab"]
catalog_ag = rrab_pop["g_absorption"]
empirical_ag = rrab_pop["A_G"]
finite = np.isfinite(catalog_ag) & np.isfinite(empirical_ag)
n_with = int(np.sum(finite))
median_offset = float(np.median(np.array(empirical_ag[finite], dtype=float) - np.array(catalog_ag[finite], dtype=float)))
print(f"\nRRab stars with finite g_absorption: {n_with}")
print(f"Median offset (empirical A_G - catalog g_absorption): {median_offset:.3f} mag")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, log10_sig]
Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 11 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, log10_sig]
Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 10 seconds.


,Class,a (slope),b (intercept),σ_color
0,RRab (N=559),+0.2483 ± 0.0422,+0.6487 ± 0.0029,0.0500 ± 0.0025
1,RRc (N=316),+0.4792 ± 0.0529,+0.4451 ± 0.0034,0.0507 ± 0.0026



RRab stars with finite g_absorption: 142867
Median offset (empirical A_G - catalog g_absorption): -0.146 mag


In [ ]:
# =====================================================================
# SECTION 04: DUST MAPPING & SFD COMPARISON
# =====================================================================
from dustmaps.config import config as dustmaps_config
import dustmaps.sfd
from dustmaps.sfd import SFDQuery
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy import stats as sp_stats

# --- Quality cut cascade from population ---
n_total = len(population)
print(f"Full catalog (population): {n_total} stars")

remaining = np.ones(n_total, dtype=bool)
cascade = []

bp_snr = population["phot_bp_mean_flux_over_error"]
cut = bp_snr > 5
n_removed = int(np.sum(remaining & ~cut))
remaining &= cut
cascade.append({"Cut": "BP SNR > 5", "Removed": n_removed, "%": f"{100*n_removed/n_total:.1f}", "Remaining": int(np.sum(remaining))})

rp_snr = population["phot_rp_mean_flux_over_error"]
cut = rp_snr > 5
n_removed = int(np.sum(remaining & ~cut))
remaining &= cut
cascade.append({"Cut": "RP SNR > 5", "Removed": n_removed, "%": f"{100*n_removed/n_total:.1f}", "Remaining": int(np.sum(remaining))})

bp_rp_pop = population["bp_rp"]
E_pop = population["phot_bp_rp_excess_factor"]
cut = (E_pop > 1.0 + 0.015 * bp_rp_pop**2) & (E_pop < 1.3 + 0.06 * bp_rp_pop**2)
n_removed = int(np.sum(remaining & ~cut))
remaining &= cut
cascade.append({"Cut": "BP/RP excess envelope", "Removed": n_removed, "%": f"{100*n_removed/n_total:.1f}", "Remaining": int(np.sum(remaining))})

sigma_e = population["sigma_E"]
cut = np.isfinite(sigma_e) & (sigma_e <= 0.15)
n_removed = int(np.sum(remaining & ~cut))
remaining &= cut
cascade.append({"Cut": "σ_E ≤ 0.15 mag", "Removed": n_removed, "%": f"{100*n_removed/n_total:.1f}", "Remaining": int(np.sum(remaining))})

e_bprp = population["E_bprp"]
cut = np.isfinite(e_bprp) & (e_bprp >= 0.0)
n_removed = int(np.sum(remaining & ~cut))
remaining &= cut
cascade.append({"Cut": "E(BP-RP) ≥ 0", "Removed": n_removed, "%": f"{100*n_removed/n_total:.1f}", "Remaining": int(np.sum(remaining))})

display(pd.DataFrame(cascade))

# --- Final sample composition ---
print(f"\nFinal reddening sample: {len(rrlyrae_clean_map)} stars ({100*len(rrlyrae_clean_map)/n_total:.1f}% of population)")
counts = Counter(rrlyrae_clean_map["best_classification"])
for cls in ["RRab", "RRc"]:
    print(f"  {cls}: {counts.get(cls, 0)}")

# --- SFD comparison ---
l_deg = rrlyrae_clean_map["l"]
b_deg = rrlyrae_clean_map["b"]
empirical = rrlyrae_clean_map["E_bprp"]

DUSTMAPS_DATA_DIR = Path("data/dustmaps-data")
DUSTMAPS_DATA_DIR.mkdir(parents=True, exist_ok=True)
dustmaps_config["data_dir"] = str(DUSTMAPS_DATA_DIR.resolve())

try:
    sfd = SFDQuery()
except FileNotFoundError:
    dustmaps.sfd.fetch()
    sfd = SFDQuery()

coords = SkyCoord(l=l_deg * u.deg, b=b_deg * u.deg, frame="galactic")
sfd_ebv = np.array(sfd(coords), dtype=float)

finite_mask = np.isfinite(sfd_ebv) & np.isfinite(empirical)

# Overall R²
x_all = sfd_ebv[finite_mask]
y_all = np.array(empirical[finite_mask], dtype=float)
r2_overall = float(sp_stats.pearsonr(x_all, y_all)[0]**2)

# Latitude-binned R²
b_abs = np.abs(np.array(b_deg, dtype=float))
lat_bins = [(0,10),(10,20),(20,30),(30,40),(40,50),(50,60),(60,70),(70,80),(80,90)]
rows = []
for lo, hi in lat_bins:
    mask = finite_mask & (b_abs >= lo) & (b_abs < hi)
    n = int(np.sum(mask))
    if n > 2:
        r2 = float(sp_stats.pearsonr(sfd_ebv[mask], np.array(empirical[mask], dtype=float))[0]**2)
    else:
        r2 = float("nan")
    label = f"|b| < {hi}°" if lo == 0 else (f"|b| > {lo}°" if hi == 90 else f"{lo}° < |b| < {hi}°")
    rows.append({"Latitude": label, "N": n, "R²": round(r2, 4)})
rows.append({"Latitude": "Overall", "N": int(np.sum(finite_mask)), "R²": round(r2_overall, 4)})
display(pd.DataFrame(rows))

# --- Regime decomposition ---
SIMILAR_SCALE_MAX = 2.0
LARGE_SFD_MIN = 10.0

similar_mask = finite_mask & (sfd_ebv <= SIMILAR_SCALE_MAX)
large_mask = finite_mask & (sfd_ebv > LARGE_SFD_MIN)

n_similar = int(np.sum(similar_mask))
n_large = int(np.sum(large_mask))
x_sim = sfd_ebv[similar_mask]
y_sim = np.array(empirical[similar_mask], dtype=float)
slope, intercept = np.polyfit(x_sim, y_sim, 1)
r2_sim = float(sp_stats.pearsonr(x_sim, y_sim)[0]**2)

print(f"\nSimilar-scale (E(B-V) ≤ {SIMILAR_SCALE_MAX}): N = {n_similar}, slope = {slope:.2f}, intercept = {intercept:.2f}, R² = {r2_sim:.3f}")
print(f"Large-SFD (E(B-V) > {LARGE_SFD_MIN}): N = {n_large}")